# 直接使用 AgentEventStream

如果不想使用 `Agent` 类，可以直接使用低层 API `agent_loop` / `agent_loop_continue`，它们返回 `AgentEventStream`，适合与自定义 UI 集成。

本 Notebook 同时演示如何写一个 **离线 mock stream**，在没有 API Key 的情况下也能跑通 Agent loop。


## 1. Mock stream 函数

`stream_fn` 需要与 `nova_ai.stream_simple` 签名一致：`(model, context, options) -> async_iterable`，并且可调用 `.result()` 获取 `AssistantMessage`。


In [1]:
from nova_ai import (
    AssistantMessage, TextContent, AssistantMessageEventStream,
    StartEvent, TextDeltaEvent, DoneEvent,
)

async def mock_stream_fn(model, context, options):
    '''离线 mock：无论用户问什么，都返回一句固定的话。'''
    # 找到最后一条用户消息
    user_text = ""
    for m in reversed(context.messages):
        if m.role == "user":
            user_text = m.content[0].text if isinstance(m.content, list) else str(m.content)
            break

    reply = f"收到你的消息：{user_text[:30]}"
    msg = AssistantMessage(
        role="assistant",
        content=[TextContent(text=reply)],
        api=model.api,
        provider=model.provider,
        model=model.id,
    )

    stream = AssistantMessageEventStream()
    stream.push(StartEvent(partial=msg))
    for ch in reply:
        stream.push(TextDeltaEvent(delta=ch, partial=msg, content_index=0))
    stream.push(DoneEvent(reason="stop", message=msg))
    return stream


## 2. 使用 agent_loop

`agent_loop` 会异步启动 loop，立即返回 `AgentEventStream`。


In [2]:
from nova_agent import agent_loop
from nova_agent.types import AgentContext, AgentLoopConfig
from nova_ai import get_model, UserMessage

model = get_model("volcengine", "deepseek-v3-2-251201")
context = AgentContext(
    system_prompt="你是一个离线测试助手。",
    messages=[],
)
config = AgentLoopConfig(model=model)

stream = agent_loop(
    prompts=[UserMessage(role="user", content="你好，世界！")],
    context=context,
    config=config,
    stream_fn=mock_stream_fn,
)

async for event in stream:
    print(f"[{event.type}]")

messages = await stream.result()
print("\n最终消息数:", len(messages))
print("最后一条:", messages[-1].content[0].text)


[agent_start]
[turn_start]
[message_start]
[message_end]
[message_start]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_update]
[message_end]
[turn_end]
[agent_end]

最终消息数: 2
最后一条: 收到你的消息：你好，世界！
